# Complex Networks Computational Session
# Random networks, Centrality measures, Markov chain processes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import networkx as nx
from scipy.linalg import eig
from scipy.sparse.csgraph import shortest_path
import pandas as pd
import seaborn as sns
import urllib.request

In [ ]:
# Function to visualize an adjacency matrix
def plot_adjacency_matrix(adj_matrix, title="Adjacency Matrix"):
    plt.figure(figsize=(6,6))
    plt.imshow(adj_matrix, cmap='gray_r', interpolation='none')
    plt.title(title)
    plt.colorbar(label="Edge Weight")
    plt.show()

# Check if A is symmetric (difference between A and A.T is zero) 
#def is_symmetric(A):
#    return np.all(A - A.T == 0)

# same with np.isclose
def is_symmetric(A):
    return np.allclose(A, A.T)

## Random graphs
### Microcanonical ensemble of networks with N nodes and M edges
Analogy with fixed energy E (fixed number of links) 

In [ ]:
N=100
M=2000

# Create n random graphs
n=100

Ms=np.zeros(n)
for i in tqdm(range(n)):
    A = np.zeros((N,N))
    for j in range(M):
        a = np.random.randint(0,N)
        b = np.random.randint(0,N)
        while a==b or A[a,b]==1:
            a = np.random.randint(0,N)
            b = np.random.randint(0,N)
        A[a,b]=1
        A[b,a]=1
    # Save number of edges
    Ms[i]=np.sum(A)/2


### Canonical ensemble of networks with N nodes (Erdos-Renyi graph)
Analogy with fixed temperature T -> the constraint is the expected value of the energy \<E\>, but i can fluctuate

In [ ]:
N=100
p=0.1
n=100

Ms=np.zeros(n)
for i in tqdm(range(n)):
    # Matrix of random numbers between 0 and 1 (uniform distribution)
    A = np.random.rand(N,N)
    # Create adjacency matrix
    A = A<p
    # Make it symmetric
    A = np.triu(A,1)
    A = A + A.T
    # Save number of edges
    Ms[i]=np.sum(A)/2

In [ ]:
plt.hist(Ms, bins=20);

In [ ]:
def erdos_renyi(N,p):
    A = np.random.rand(N,N)
    A = A<p
    A = np.triu(A,1)
    A = A + A.T
    return A


In [ ]:
# Density
N=100
A=erdos_renyi(N,1)
M=np.sum(A)/2
print('Number of edges:',M)
density = M/(N*(N-1)/2)
print(density)
plot_adjacency_matrix(A, "Erdos-Renyi Graph")

In [ ]:
# Ratio between standard deviation and mean number of edges goes to zero as N increases (termodynamic limit)
n=100
Ns=[10,25,50,75,100,500,1000]
ratios=[]
for N in Ns:
    Ms=np.zeros(n)
    for i in range(n):
        A = erdos_renyi(N,0.1)
        M=np.sum(A)/2
        Ms[i]=M
    ratios.append(np.std(Ms)/np.mean(M))

plt.plot(Ns,ratios,'o-')
plt.xlabel('N')
plt.ylabel('st.dev / mean number of edges')

## Stochastic Block Model (SBM)

In [ ]:
def generate_sbm(sizes, p_in, p_out):
    n = sum(sizes)
    adj = np.zeros((n, n))
    # Intra-community links
    start = 0
    for i, size in enumerate(sizes):
        end = start + size
        adj[start:end, start:end] = np.random.rand(size, size) < p_in[i]
        #simmetrize
        adj[start:end, start:end] = np.triu(adj[start:end, start:end],1)
        adj[start:end, start:end] = adj[start:end, start:end] + adj[start:end, start:end].T
        start = end
    # Inter-community links
    for i in range(len(sizes)):
        for j in range(i+1, len(sizes)):
            start_i, end_i = sum(sizes[:i]), sum(sizes[:i+1])
            start_j, end_j = sum(sizes[:j]), sum(sizes[:j+1])
            adj[start_i:end_i, start_j:end_j] = np.random.rand(end_i-start_i, end_j-start_j) < p_out[i,j]
            adj[start_j:end_j, start_i:end_i] = adj[start_i:end_i, start_j:end_j].T
    return adj

In [ ]:
sizes = [100, 30, 20]  # Three communities of size 5
p_in = [0.3, 0.4, 0.5]  # High intra-cluster density
p_out = np.array([[0, 0.1, 0.05], 
                  [0.1, 0, 0.1],
                  [0.05, 0.1, 0]])  # Sparse inter-cluster density

# N.B. actually it is considering only the upper triangular part of the block connectivity matrix

sbm_adj = generate_sbm(sizes, p_in, p_out)
plot_adjacency_matrix(sbm_adj, "Stochastic Block Model (SBM)")

### Giant component transition on ER graphs

In [ ]:
N=200

# Matrix of random numbers between 0 and 1 (uniform distribution)
A = np.random.rand(N,N)
ps=np.logspace(-4,0,20)
Ms=np.zeros(len(ps))
GCs=np.zeros(len(ps))
for i,p in tqdm(enumerate(ps)):
    # Create adjacency matrix
    Ai = A<p
    # Make it symmetric
    Ai = np.triu(Ai,1)
    Ai = Ai + Ai.T
    # Create a graph
    G = nx.Graph(Ai)
    
    components = list(nx.connected_components(G))
    # Get the giant component
    comp_sizes = [len(comp) for comp in components]
    GCs[i] = max(comp_sizes)

    

In [ ]:
# Plot p and the size of the giant component
plt.plot(ps,GCs,'-o')
plt.xscale('log')
plt.xlabel('p')
plt.ylabel('Size of the giant component')

# Centrality measures

In [ ]:
# Function to visualize an adjacency matrix
def plot_adjacency_matrix(adj_matrix, title="Adjacency Matrix"):
    plt.figure(figsize=(6,6))
    plt.imshow(adj_matrix, cmap='gray_r', interpolation='none')
    plt.title(title)
    plt.colorbar(label="Edge Weight")
    plt.show()

# Check if A is symmetric (difference between A and A.T is zero) without using numpy's isclose function
def is_symmetric(A):
    return np.all(A - A.T == 0)

def erdos_renyi(N,p,symmetric=True):
    A = np.random.rand(N,N)
    A = A<p
    if symmetric:
        A = np.triu(A,1)
        A = A + A.T
    return A

def generate_sbm(sizes, p_in, p_out,symmetric=True):
    n = sum(sizes)
    adj = np.zeros((n, n))
    # Intra-community links
    start = 0
    for i, size in enumerate(sizes):
        end = start + size
        adj[start:end, start:end] = np.random.rand(size, size) < p_in[i]
        if symmetric:
            #symmetrize
            adj[start:end, start:end] = np.triu(adj[start:end, start:end],1)
            adj[start:end, start:end] = adj[start:end, start:end] + adj[start:end, start:end].T
        start = end
    # Inter-community links
    for i in range(len(sizes)):
        for j in range(i+1, len(sizes)):
            start_i, end_i = sum(sizes[:i]), sum(sizes[:i+1])
            start_j, end_j = sum(sizes[:j]), sum(sizes[:j+1])
            adj[start_i:end_i, start_j:end_j] = np.random.rand(end_i-start_i, end_j-start_j) < p_out[i,j]
            if symmetric:
                adj[start_j:end_j, start_i:end_i] = adj[start_i:end_i, start_j:end_j].T
            else:
                adj[start_j:end_j, start_i:end_i] = np.random.rand(end_j-start_j, end_i-start_i) < p_out[i,j]
    return adj

# Build some graphs

In [ ]:
url="https://raw.githubusercontent.com/FraDurazzi/CNLab/refs/heads/master/data/disease_disease_projection_lcc.graphml"
with urllib.request.urlopen(url) as f:
    G_real = nx.read_graphml(f)

In [ ]:
# unique_classes is the set of unique values in the 'class' attribute of the nodes in G_real
unique_classes = set(nx.get_node_attributes(G_real, 'class').values())
class_colors = {cls: plt.cm.tab20(i) for i, cls in enumerate(unique_classes)}
node_colors = [class_colors[G_real.nodes[node]['class']] for node in G_real.nodes()]
labels = {node: G_real.nodes[node]['name'] for node in G_real.nodes()}
nx.draw(G_real, node_color=node_colors, with_labels=True, font_size=4,labels=labels)

In [ ]:
A_real = nx.adjacency_matrix(G_real).todense()
plot_adjacency_matrix(A_real, "Real Adjacency Matrix")
print(G_real)

In [ ]:
is_symmetric(A_real)

In [ ]:
# density of the graph
1188/(516*515/2)

In [ ]:
A_ER=erdos_renyi(516,0.0089)
plot_adjacency_matrix(A_ER)
G_ER=nx.Graph(A_ER)
print(G_ER)

In [ ]:
A_SBM=generate_sbm([300,200,100],[0.05,0.07,0.07],
                   np.array([[np.nan,0.01,0.005],
                             [np.nan,np.nan,0.015],
                             [np.nan,np.nan,np.nan]]))
plot_adjacency_matrix(A_SBM)
G_SBM=nx.Graph(A_SBM)
print(G_SBM)

# Node centrality measures

### Connectivity degree

In [ ]:
# number of neighbors of each node (local measure)
def degree_centrality(A):
    in_degree = np.sum(A, axis=0)
    out_degree = np.sum(A, axis=1)
    return in_degree, out_degree

In [ ]:
plt.hist(degree_centrality(A_real)[1], bins=50,edgecolor='black')
plt.xlabel("Degree")

### Betweenness centrality
Fraction of all the shortest paths passing through this node (w.r.t. all the shortest paths between each pair of nodes)

In [ ]:
# Create a small toy network
G = nx.Graph()
edges = [(1, 2), (1, 3), (2, 4), (3, 4), (4, 5), (5, 6), (4, 6)]
G.add_edges_from(edges)

# Compute betweenness centrality
betweenness = nx.betweenness_centrality(G, normalized=False)

# Draw the graph
pos = nx.spring_layout(G, seed=42)  # Fix layout for consistency
node_size = [1000 * betweenness[n] + 300 for n in G.nodes()]  # Scale node size by betweenness

plt.figure(figsize=(6, 4))
nx.draw(G, pos, with_labels=True, node_size=node_size, node_color='lightblue', edge_color='gray')
#nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f"{betweenness[u]:.2f}" for u, v in G.edges()})

plt.title("Toy Network with Betweenness Centrality")
plt.show()

In [ ]:
betweenness

### Closeness centrality
Reciprocal of the sum of shortest paths to all other nodes

In [ ]:
# shortest_path returns the length of the shortest path between all pairs of nodes
shortest_path(nx.adjacency_matrix(G))

In [ ]:
# closeness of the node to all others (reachability): global measure
def closeness_centrality(A):
    n=A.shape[0]
    sp = shortest_path(A, directed=True, unweighted=True)
    return (n-1) / np.sum(sp, axis=1)

In [ ]:
closeness=closeness_centrality(nx.adjacency_matrix(G).todense())
closeness

In [ ]:
node_size = [1000 * closeness[i]  for i in range(len(G.nodes()))]  # Scale node size by betweenness

plt.figure(figsize=(6, 4))
nx.draw(G, pos, with_labels=True, node_size=node_size, node_color='lightblue', edge_color='gray')

plt.title("Toy Network with Closeness Centrality")
plt.show()

### Correlation between measures

In [ ]:
G= G_real.copy()
x= nx.closeness_centrality(G).values()
y= nx.betweenness_centrality(G).values()

plt.scatter(x,y)


### Exercise
Color the 2D scatter plot according to some label (block in SBM, disease class in diseasome...)

# Transition matrix and Markov chains

In [ ]:
A_SBM=generate_sbm([50,30,20],[0.2,0.4,0.4],
                     np.array([[np.nan,0.0015,0.002],
                              [np.nan,np.nan,0.0017],
                              [np.nan,np.nan,np.nan]]))
plot_adjacency_matrix(A_SBM)

In [ ]:
# Generate the (M+N)x(M+N) adjacency matrix of a random bipartite graph
# with zero diagonal blocks and rectangular off-diagonal blocks.
def bipartite_adjacency_matrix(M, N, p, dtype=float):
    biadjacency = (np.random.rand(M, N) < p).astype(dtype)
    adjacency = np.zeros((M + N, M + N), dtype=dtype)
    adjacency[:M, M:] = biadjacency
    adjacency[M:, :M] = biadjacency.T
    return adjacency


A_BP = bipartite_adjacency_matrix(500, 200, 0.05)
plot_adjacency_matrix(A_BP)

In [ ]:
T = A_BP/np.sum(A_BP,axis=0)
plot_adjacency_matrix(T)
print(sum(T[:,0]))

In [ ]:
def markov_chain(T, steps=50, start_node=0):
    n = T.shape[0]
    state = np.zeros(n)
    state[start_node] = 1
    history=[state]
    for _ in range(steps):
        state = T @ state
        history.append(state)
    return np.array(history)

In [ ]:
state_history=markov_chain(T,steps=20,start_node=2)
state_history

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(state_history.T>0,cmap='gray_r', cbar=False);

In [ ]:
plt.bar(np.arange(len(T)),state_history[-1])

### Perron-Frobenius
If T is *irreducible* and *aperiodic*, it has an unique eigenvalue equal to 1. Thus, it exists a stationary state (_T x =1 x_  with x being the eigenvector).

*Irreducible*: no disconnected components (each node reachable by any other)

*Aperiodic*: there is no fixed cycle length that constrains the return times. You can return to a state at irregular times, not only at multiples of some fixed number.

In [ ]:
eigvals, eigvecs = eig(T, left=False,right=True)
max_idx = np.argmax(eigvals)
print(max_idx)
stationary_eigenvector = np.abs(eigvecs[:, max_idx]) / np.sum(np.abs(eigvecs[:, max_idx]))

In [ ]:
eigvals[max_idx]

In [ ]:
minus_idx = np.argmin(np.abs(eigvals + 1))
minus_eigenvalue = eigvals[minus_idx]
alternating_mode = np.real(eigvecs[:, minus_idx])
alternating_mode = alternating_mode / np.max(np.abs(alternating_mode))

print(minus_eigenvalue)

In [ ]:
plt.figure(figsize=(12,4))
plt.bar(np.arange(len(T)), alternating_mode)
plt.axvline(500 - 0.5, color='black', linestyle='--')
plt.title("Eigenvector associated with the eigenvalue near -1")
plt.xlabel("Node index")
plt.ylabel("Signed component")
plt.show()

In [ ]:
plt.bar(np.arange(len(T)),stationary_eigenvector)

In the end, the in-degree is the only relevant factor, no matters where you started (ergodicity).

In [ ]:
plt.scatter(state_history[-1], degree_centrality(A_ER)[0])

# Random walks

In [ ]:
G = nx.Graph(A_SBM)

def random_walk(G, steps=50, start_node=0):
    """Simple random walk on a graph.
    
    At each step, the walker moves to a uniformly random neighbor.
    
    Inputs:
        - G: networkx graph
        - steps: total number of steps
        - start_node: initial node
    
    Outputs:
        - history: (steps+1) x n array, where history[t,i]=1 if walker is at node i at time t
    """
    n = G.number_of_nodes()
    current_node = start_node
    history = [np.zeros(n)]
    history[0][current_node] = 1
    
    for _ in range(steps):
        neighbors = list(G.neighbors(current_node))
        current_node = np.random.choice(neighbors)
        state = np.zeros(n)
        state[current_node] = 1
        history.append(state)
    
    return np.array(history)

In [ ]:
state_history = random_walk(G, steps=1000, start_node=2)

In [ ]:
sns.heatmap(state_history.T>0,cmap='gray_r', cbar=False);

### Exercise: Proximity score via damped random walk (Personalized PageRank)

A simple random walk converges slowly. Add a **damping factor** to model a walker that sometimes "teleports" back to the starting node.

**Step 1:** Modify `random_walk()` to add a damping parameter `d` (e.g., d=0.8):
- At each step, with probability `1-d`, teleport back to `start_node`
- With probability `d`, continue to a random neighbor as usual

**Step 2:** Run this damped walk for many steps (e.g., 10000), then compute:
- Visit count: how many times each node was visited
- Normalize to get a **proximity score** (probability of being at each node)

**Step 3:** Compare:
- The proximity score from your damped walk
- The stationary distribution of the transition matrix `T` (our `stationary_eigenvector`)
- Note: your damped walk computes an approximation of **personalized PageRank** with restart at `start_node`

**Bonus:** Try different values of d and starting nodes. What changes?

In [ ]:
# SOLUTION TEMPLATE (uncomment and complete)
def damped_random_walk(G, steps=5000, start_node=0, damping=0.8):
    """Random walk with teleportation (damping).
    
    With probability 1-damping, teleport back to start_node.
    With probability damping, take a random step as usual.
    """
    # TODO: Implement this function
    # Hint: use np.random.choice() to decide whether to teleport
    pass

# Once implemented, compute proximity score:
# proximity_score = damped_random_walk(G, steps=5000, start_node=2, damping=0.8)
# Could visualize and compare with stationary_eigenvector from the SBM case